# LibAR 검출기 v3 재학습 (밀집 라벨)

**목적:** dataset_v2(원거리 1080p, 장당 ~14박스 희소 라벨 → mAP50 0.50 정체)를
shelf_4558 기반 **밀집 라벨 dataset_v3**로 교체 재학습. 박스 걸침 방지 라벨 규칙 적용.

**사용법:** 런타임 → GPU(T4) 선택 → 아래 셀 순서대로 실행.
업로드 파일 2개: `dataset_v3.zip`, (선택) 기존 `best.pt`

In [ ]:
!pip install -q ultralytics
import ultralytics; ultralytics.checks()

In [ ]:
# dataset_v3.zip 업로드 (+ 선택: 기존 best.pt 도 함께 업로드하면 전이학습)
from google.colab import files
up = files.upload()
!unzip -oq dataset_v3.zip
!ls dataset_v3/images/train dataset_v3/images/val

In [ ]:
from ultralytics import YOLO
import os
# 기존 best.pt 업로드했으면 전이학습, 아니면 yolo26n부터
base = 'best.pt' if os.path.exists('best.pt') else 'yolo26n.pt'
print('init from:', base)
model = YOLO(base)
model.train(
    data='dataset_v3/data.yaml',
    epochs=80, imgsz=1536, batch=4, patience=30,
    # 텍스트 방향 보존: 좌우반전 금지. 촬영각/조명 증강은 적극적으로
    fliplr=0.0, flipud=0.0,
    degrees=8, translate=0.08, scale=0.25, shear=2, perspective=0.0003,
    hsv_h=0.01, hsv_s=0.4, hsv_v=0.35,
    mosaic=0.5, close_mosaic=15,
)

In [ ]:
# 검증 지표 확인 (목표: mAP50 0.50 → 0.75+)
m = YOLO('runs/detect/train/weights/best.pt')
m.val(data='dataset_v3/data.yaml', imgsz=1536)

In [ ]:
# 결과 다운로드 → 로컬 libar-sample/best_v3.pt 로 저장
from google.colab import files
files.download('runs/detect/train/weights/best.pt')